### TEST 04D · XGB con PCs del gradiente REAL (dominio pequeño)
Idea: si las PCs del gradiente real entran en el estado del PVAR, la relación presión–gradiente pasa a ser una correlación lineal que el VAR sí reproduce. Antes de montar el PVAR, comprobamos cuánto skill tiene el XGB con 30/20/10 PCs de presión + 10 PCs del gradiente real, frente al actual (30 presión + 10 reconstruidas).

Guarda: `outputs/grad_rec/pcs_grad_real_6h_wmed.csv` y un modelo por configuración en `outputs/xgb_p102_wmed_{config}/`. No toca nada existente.

In [1]:
import os, json, pickle

import pandas as pd
import numpy as np
import xarray as xr

from xgboost import XGBRegressor
from sklearn.model_selection import GroupKFold
from scipy.signal import find_peaks
from utils.pit.piece_wise_marginal import from_normal

from bluemath_tk.core.io import load_model

#### Load Data

In [2]:
point = 'p102'

pcs_msl = pd.read_csv('outputs/pca/pcs_msl_30_6h_wmed.csv', index_col=0, parse_dates=True)
pcs_grec = pd.read_csv('outputs/grad_rec/pcs_grad_rec_6h_wmed.csv', index_col=0, parse_dates=True)

#### Objetivo: Z (PIT + armónica, 00_PIT), media móvil de 6 h centrada, a 00/06/12/18
waves_df_z = pd.read_csv(f'outputs/marginal/z_bulk_{point}.csv', index_col=0, parse_dates=True)
waves_df_z_6h = waves_df_z.rolling('6h', center=True).mean()
waves_df_z_6h = waves_df_z_6h[waves_df_z_6h.index.hour.isin([0, 6, 12, 18])].dropna()

pit = pickle.load(open(f'outputs/marginal/transformacion_{point}.pkl', 'rb'))

era5 = xr.open_zarr('data/era5_mslp_grad_1985_2026_1deg_hourly.zarr').rename({'valid_time': 'time'})
era5 = era5.sel(time=era5.time.dt.hour.isin([0, 6, 12, 18]))
era5 = era5.sel(latitude=slice(48, 30), longitude=slice(-5, 20))                     # dominio pequeño

pca_grad = load_model('outputs/grad_rec/pca_grad_wmed.pkl')

#### PCs del gradiente real (proyección sobre la misma PCA del gradiente que usa el 03)

In [3]:
f_real = 'outputs/grad_rec/pcs_grad_real_6h_wmed.csv'
N_GR = 10

if os.path.exists(f_real):
    pcs_greal = pd.read_csv(f_real, index_col=0, parse_dates=True)
else:
    out = []
    for a in range(1940, 2027, 10):                          # por bloques de 10 años
        g = era5[['grad2']].sel(time=slice(str(a), str(a + 9))).load()
        if g.time.size == 0:
            continue
        out.append(pd.DataFrame(pca_grad.transform(data=g)['PCs'].values[:, :N_GR], index=g.time.values,
                                columns=[f'grad_PC{k + 1}' for k in range(N_GR)]))
    pcs_greal = pd.concat(out)
    pcs_greal.index.name = 'time'
    pcs_greal.to_csv(f_real, float_format='%.5f')

print(pcs_greal.shape, pcs_greal.index[0], pcs_greal.index[-1])

#### Reconstruido vs real (todo el periodo) y asimetría de cada PC
t = pcs_greal.index.intersection(pcs_grec.index)
pd.DataFrame({'corr rec–real': [np.corrcoef(pcs_grec.loc[t].iloc[:, k], pcs_greal.loc[t].iloc[:, k])[0, 1] for k in range(N_GR)],
              'asimetría real': pcs_greal.skew().values,
              'asimetría rec': pcs_grec.skew().values}, index=pcs_greal.columns).round(2)

Using 450 out of 450 available variables 
If this is originated by using few times, please check 'nan_threshold_to_drop' parameter in fit method
Using 450 out of 450 available variables 
If this is originated by using few times, please check 'nan_threshold_to_drop' parameter in fit method
Using 450 out of 450 available variables 
If this is originated by using few times, please check 'nan_threshold_to_drop' parameter in fit method
Using 450 out of 450 available variables 
If this is originated by using few times, please check 'nan_threshold_to_drop' parameter in fit method
Using 450 out of 450 available variables 
If this is originated by using few times, please check 'nan_threshold_to_drop' parameter in fit method
Using 450 out of 450 available variables 
If this is originated by using few times, please check 'nan_threshold_to_drop' parameter in fit method
Using 450 out of 450 available variables 
If this is originated by using few times, please check 'nan_threshold_to_drop' parameter

(126864, 10) 1940-01-01 00:00:00 2026-10-31 18:00:00


,corr rec–real,asimetría real,asimetría rec
grad_PC1,0.99,2.13,2.27
grad_PC2,0.97,0.32,0.89
grad_PC3,0.97,0.50,0.65
grad_PC4,0.96,-0.61,0.21
grad_PC5,0.93,0.23,1.00
grad_PC6,0.95,0.30,0.04
grad_PC7,0.92,-0.19,0.63
grad_PC8,0.90,0.80,0.28
grad_PC9,0.82,0.53,-1.09
grad_PC10,0.85,0.45,0.69


#### Configuraciones a comparar

In [4]:
CONFIGS = {
    'actual':  pcs_msl.join(pcs_grec),                          # 30 presión + 10 gradiente reconstruido
    'p30_g10': pcs_msl.join(pcs_greal),                         # 30 presión + 10 gradiente real
    'p20_g10': pcs_msl.iloc[:, :20].join(pcs_greal),            # 20 + 10  → candidato para el PVAR
    'p10_g10': pcs_msl.iloc[:, :10].join(pcs_greal),            # 10 + 10
}

lags = [0, 6, 12, 18, 24, 30, 36]
xgb_params = dict(n_estimators=300, learning_rate=0.05, max_depth=3, min_child_weight=200,
                  subsample=0.8, colsample_bytree=0.8, tree_method='hist', n_jobs=-1, random_state=0)

In [5]:
def features(d):
    Xs = pd.concat({f'l{l}': d.shift(freq=pd.Timedelta(hours=l)) for l in lags}, axis=1).reindex(d.index)
    Xs.columns = [f'{c}_l{l[1:]}' for l, c in Xs.columns]
    return Xs.dropna()

def hs(z):
    zz = waves_df_z.loc[z.index].copy()
    zz['VHM0'] = z.values
    z0 = pit['harm_z'].inverse_transform(zz)['VHM0'].clip(-6.36, 6.36)
    return pd.Series(pit['marg']['VHM0'].ppf(from_normal(z0).values), index=z.index)

def evaluar(nombre, data_pcs):
    X_all = features(data_pcs)
    idx = X_all.index.intersection(waves_df_z_6h.index)
    X, y = X_all.loc[idx], waves_df_z_6h.loc[idx, 'VHM0']
    tr = idx < '2016-01-01'

    #### OOF por años (1985–2015)
    f_oof = pd.Series(np.nan, index=X[tr].index)
    for i_fit, i_val in GroupKFold(n_splits=5).split(X[tr], y[tr], groups=X[tr].index.year):
        m = XGBRegressor(**xgb_params).fit(X[tr].iloc[i_fit], y[tr].iloc[i_fit])
        f_oof.iloc[i_val] = m.predict(X[tr].iloc[i_val])

    #### Modelo final y test 2016–2025
    model = XGBRegressor(**xgb_params).fit(X[tr], y[tr])
    f_test = pd.Series(model.predict(X[~tr]), index=X.index[~tr])

    #### Extremos en test, en metros
    hs_obs, hs_f = hs(y[~tr]), hs(f_test)
    picos = hs_obs.index[find_peaks(hs_obs.values, height=hs_obs.quantile(0.99), distance=12)[0]]
    ratio = pd.Series([hs_f.loc[t - pd.Timedelta('24h'):t + pd.Timedelta('24h')].max() / hs_obs.loc[t] for t in picos], index=picos)
    top10 = hs_obs.loc[picos].nlargest(10).index

    #### Guardar
    out_dir = f'outputs/xgb_{point}_wmed_{nombre}'
    os.makedirs(out_dir, exist_ok=True)
    model.save_model(f'{out_dir}/xgb_final.json')
    with open(f'{out_dir}/xgb_features.json', 'w') as f:
        json.dump({'columns': list(X.columns), 'lags_h': lags, 'params': xgb_params,
                   'train': ['1985-01-01', '2015-12-31']}, f, indent=1)

    return {'n PCs': data_pcs.shape[1],
            'corr OOF': f_oof.corr(y[tr]), 'corr test': f_test.corr(y[~tr]),
            'corr diaria test': f_test.resample('D').mean().corr(y[~tr].resample('D').mean()),
            'pico f/obs test': ratio.mean(), '10 mayores': ratio.loc[top10].mean(),
            'Juliette f [m]': hs_f.loc['2023-02-25':'2023-03-03'].max()}

In [6]:
resultados = {}
for nombre, d in CONFIGS.items():
    resultados[nombre] = evaluar(nombre, d)
    print(nombre, 'hecho')

pd.DataFrame(resultados).T.round(3)

actual hecho
p30_g10 hecho
p20_g10 hecho
p10_g10 hecho


,n PCs,corr OOF,corr test,corr diaria test,pico f/obs test,10 mayores,Juliette f [m]
actual,40.0,0.848,0.838,0.866,0.815,0.822,4.654
p30_g10,40.0,0.846,0.837,0.864,0.793,0.798,5.110
p20_g10,30.0,0.845,0.834,0.861,0.801,0.818,5.178
p10_g10,20.0,0.837,0.825,0.852,0.801,0.822,5.206
